# ACTIVE-TB-S: Full Five-Seed Post-Freeze Evaluation

This phase receives hash-frozen testbenches from five predeclared synthesis seeds plus VerilogEval golden RTL, official testbenches, and the released CorrectBench mutant bank. Generated suites cannot be modified here.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
markers=list(INPUT.rglob('active_tb_s_evaluation_full_v1/EVALUATION_BOUNDARY.json'))
if len(markers)==1: SOURCE=markers[0].parent
else:
 archives=list(INPUT.rglob('active_tb_s_evaluation_full_v1.zip')); assert len(archives)==1,archives
 attached=WORK/'attached_full_evaluation_source'
 with zipfile.ZipFile(archives[0]) as zf: zf.extractall(attached)
 SOURCE=attached/'active_tb_s_evaluation_full_v1'
PROJECT=WORK/'active_tb_s_evaluation_full_project'
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(SOURCE/'project',PROJECT)
# Kaggle recursively expands archives inside an uploaded dataset, so the frozen
# bundle may arrive either as the original zip or already unpacked as a directory.
frozen_zip=SOURCE/'active_tb_s_frozen_full_v1.zip'
frozen_dir=SOURCE/'active_tb_s_frozen_full_v1'
if frozen_zip.exists():
 FROZEN=WORK/'active_tb_s_frozen_full_v1'
 with zipfile.ZipFile(frozen_zip) as zf: zf.extractall(FROZEN)
elif frozen_dir.is_dir(): FROZEN=frozen_dir
else: raise FileNotFoundError(f'No frozen bundle at {frozen_zip} or {frozen_dir}')
assert sorted(x.name for x in FROZEN.glob('seed_*'))==['seed_017','seed_029','seed_043','seed_059','seed_073'],sorted(x.name for x in FROZEN.glob('seed_*'))
print('Frozen bundle:',FROZEN)
OUT=WORK/'active_tb_s_evaluation_full_v1'
CHECKPOINT=WORK/'active_tb_s_evaluation_full_checkpoint.zip'
resume=list(INPUT.rglob('active_tb_s_evaluation_full_checkpoint.zip'))
if len(resume)>1: raise RuntimeError(resume)
if OUT.exists(): shutil.rmtree(OUT)
if resume:
 with zipfile.ZipFile(resume[0]) as zf: zf.extractall(OUT)
else: OUT.mkdir()
print('Evaluation source:',SOURCE,'resume:',resume)


In [ ]:
boundary=json.loads((SOURCE/'EVALUATION_BOUNDARY.json').read_text())
assert boundary['frozen_before_golden_access'] is True
assert boundary['task_count']==156 and len(boundary['seeds'])==5
for seed in boundary['seeds']:
 manifest=json.loads((FROZEN/f'seed_{seed:03d}'/'FROZEN_BEFORE_EVALUATION.json').read_text())
 assert manifest['generation_input_fields']==['task_id','description','header']
 assert manifest['random_seed']==seed
print('FULL_POST_FREEZE_PROVENANCE_AUDIT_PASS:',boundary['task_count'],'tasks',boundary['seeds'])


In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'],check=True)
for seed in boundary['seeds']:
 seed_out=OUT/f'seed_{seed:03d}'; seed_out.mkdir(parents=True,exist_ok=True)
 cmd=[sys.executable,str(PROJECT/'run_spec_only_evaluation.py'),
  '--frozen-root',str(FROZEN/f'seed_{seed:03d}'/'frozen'),
  '--tasks',str(SOURCE/'HDLBits_full.jsonl'),
  '--mutants',str(SOURCE/'HDLBits_mutants_full.jsonl'),
  '--output',str(seed_out),'--timeout','30','--resume']
 subprocess.run(cmd,check=True,cwd=PROJECT)
 shutil.make_archive(str(CHECKPOINT.with_suffix('')),'zip',OUT)
 print('Completed evaluation seed',seed,'checkpoint bytes',CHECKPOINT.stat().st_size)


In [ ]:
import numpy as np, pandas as pd
aggregates=[]; per_tasks=[]
for seed in boundary['seeds']:
 aggregate=pd.read_csv(OUT/f'seed_{seed:03d}'/'spec_only_eval2_aggregate.csv'); aggregate['seed']=seed; aggregates.append(aggregate)
 per_task=pd.read_csv(OUT/f'seed_{seed:03d}'/'spec_only_eval2_per_task.csv'); per_task['seed']=seed; per_tasks.append(per_task)
aggregate=pd.concat(aggregates,ignore_index=True); per_task=pd.concat(per_tasks,ignore_index=True)
aggregate.to_csv(OUT/'spec_only_eval2_aggregate_all_seeds.csv',index=False)
per_task.to_csv(OUT/'spec_only_eval2_per_task_all_seeds.csv',index=False)
metric_cols=['synthesis_success_rate','eval1_pass_at_1','eval2_exact_pass_at_1','eval2_loose_pass_at_1_0_8','raw_mutant_rejection_rate']
rows=[]
for method,group in aggregate.groupby('method'):
 for metric in metric_cols:
  values=group[metric].astype(float).to_numpy(); mean=float(values.mean())
  half=float(2.776*values.std(ddof=1)/np.sqrt(len(values))) if len(values)>1 else 0.0
  rows.append({'method':method,'metric':metric,'mean':mean,'ci95_low':mean-half,'ci95_high':mean+half,'seeds':len(values)})
ci=pd.DataFrame(rows); ci.to_csv(OUT/'spec_only_method_ci.csv',index=False)
print(ci.to_string(index=False))

# Paired task-level bootstrap: typed/formal versus the strongest pilot control.
rng=np.random.default_rng(20260901); typed='typed_formal_active'; control='single_oracle'
wide=per_task.pivot_table(index=['seed','task_id'],columns='method',values=['eval1_pass','eval2_exact_pass','eval2_loose_pass_0_8','eval2_ratio'],aggfunc='first')
boot=[]
for metric in ['eval1_pass','eval2_exact_pass','eval2_loose_pass_0_8','eval2_ratio']:
 pairs=wide[metric][[typed,control]].dropna().astype(float).to_numpy(); observed=float((pairs[:,0]-pairs[:,1]).mean())
 samples=[]
 for _ in range(10000):
  idx=rng.integers(0,len(pairs),len(pairs)); samples.append(float((pairs[idx,0]-pairs[idx,1]).mean()))
 boot.append({'metric':metric,'typed_minus_single':observed,'ci95_low':float(np.quantile(samples,.025)),'ci95_high':float(np.quantile(samples,.975)),'paired_rows':len(pairs)})
bootstrap=pd.DataFrame(boot); bootstrap.to_csv(OUT/'typed_vs_single_paired_bootstrap.csv',index=False)
print(bootstrap.to_string(index=False))
archive=WORK/'active_tb_s_evaluation_full_v1.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',OUT)
print('Full evaluation checkpoint:',archive,archive.stat().st_size,'bytes')
